Notebook Activity - 3
- Getting comfortable with LangChain
- Building Sequential Chains using LangChain
- Understanding the concept of Prompt Template

In [14]:
!pip -q install -U \langchain \langchain-openai \langchain-community \openai \python-dotenv \pypdf \pandas \tabulate \openpyxl \tiktoken

In [15]:
import os #for creating a virtual env
from google.colab import files #load files from my local system
from langchain_openai import ChatOpenAI #ChatOpenAI is needed for configuring the LLM
from langchain_core.prompts import ChatPromptTemplate #Useful for creating sequential prompt templates
from langchain_core.output_parsers import StrOutputParser #output generated should be like a JSON outuput - well structured in nature
from langchain_community.document_loaders import PyPDFLoader #our source of data is a PDF file - Annual Report

In [16]:
os.environ["OPENAI_API_KEY"]="YOUR API"
llm=ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.2
)

In [17]:
loader=PyPDFLoader("/content/My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")
documents=loader.load()

report = ""
for page in documents:
  report += page.page_content+'\n'

print("Total Pages:",len(documents))
print(report[:1000]) #first 1000 tokens are sliced and displayed here

Total Pages: 9
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable

In [18]:
#Building the First Chain - for creating an executive summary out of the Annual Report
summary_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior Management Consultant, your job is to read the following annual report
    and prepare a professional executive summary out of it.

    In the summary, include -
    - Company Profile
    - Financial Performance
    - Operational Highlights
    - Major Risks
    - Future Outlook

    annual report {report}
    """
)

In [19]:
summary_chain=(
    summary_prompt | llm | StrOutputParser()
)

In [20]:
summary=summary_chain.invoke(
    {"report":report}
)

print(summary) #this summary generated as o/p from Chain 1 will be used as input in Chain 2

**Executive Summary: AxiOraa Ltd. Annual Report 2025**

---

### Company Profile  
AxiOraa Ltd. is a global AI-powered consulting and analytics firm specializing in delivering AI strategy, data engineering, cloud modernization, and Generative AI (GenAI) solutions. Serving key industries such as banking, insurance, healthcare, manufacturing, and the public sector, AxiOraa operates primarily across APAC, Europe, and North America. Its core business segments include Strategy Consulting, AI Engineering, Managed Analytics, and Intelligent Automation. The company leverages proprietary AI accelerators and strategic partnerships to enhance delivery efficiency and drive enterprise AI adoption.

---

### Financial Performance  
In FY2025, AxiOraa Ltd. demonstrated robust financial growth with revenue increasing by 21.1% from USD 2.18 billion to USD 2.64 billion. Gross margin improved to 44.2% from 41.5%, reflecting enhanced operational efficiency. EBITDA margin stood at 22.8%, with EBITDA totali

In [21]:
#Chain 2 - will be used to review the risks
risk_prompt=ChatPromptTemplate.from_template(
    """
    You are a Risk Advisory Partner, review the attached summary of annual report
    and prepare a detailed enterprise risk review report
    executive summary {summary}

    In the Risk Review Report include -
    - Description of each risk
    - Business impact due to the risk
    - Likelihood of the risk taking place
    - Severity of the risk

    """
)

In [22]:
risk_chain=(
    risk_prompt | llm | StrOutputParser()
)

In [23]:
risk=risk_chain.invoke(
    {"summary":summary}
)

print(risk) #this risk generated as o/p from Chain 2 will be used as input in Chain 3

**Enterprise Risk Review Report**  
**AxiOraa Ltd. – FY2025 Annual Report Analysis**  

---

### Introduction  
This Enterprise Risk Review Report provides a detailed assessment of the key risks identified in AxiOraa Ltd.’s FY2025 Annual Report. The analysis includes a description of each risk, its potential business impact, likelihood of occurrence, and severity to inform risk mitigation priorities and strategic decision-making.

---

### 1. Cybersecurity Threats and Evolving Regulatory Uncertainty  

**Description:**  
As a global AI-powered consulting and analytics firm, AxiOraa handles vast amounts of sensitive client data and proprietary AI models. Cybersecurity threats include data breaches, ransomware attacks, and intellectual property theft. Additionally, regulatory environments around data privacy, AI usage, and cross-border data flows are rapidly evolving, creating compliance challenges.

**Business Impact:**  
- Data breaches could lead to loss of client trust, legal penalti

Activity -

- Once we have the summary of the Annual Report and Risks involved available, we can next try to generate Strategic Growth Opportunities for the Client by supplying the summary of the annual report and risks as input

Notebook Activity - 4
- Building an end to end RAG Pipeline

In [24]:
from langchain_openai import OpenAIEmbeddings #this will help in creating embeddings after chunking operation
from langchain_community.vectorstores import FAISS #this is our preferred Vector DB
from langchain_text_splitters import RecursiveCharacterTextSplitter #helpful in performing chunking

In [25]:
#just the way we configure our LLM, we will configure our Embedding Model as well
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [27]:
#creating chunks
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000, #each chunk will have 1000 tokens
    chunk_overlap=200 #each chunk will have an overlap of 200 tokens
)

chunks=text_splitter.split_documents(documents)

print(f"Total Number of Chunks Created:{len(chunks)}")

Total Number of Chunks Created:44


- Try out with chunk_size = 2500, chunks = 17
- Try out with chunk_size = 1000, chunks = 44

Which one is better ?

In [29]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.5 MB/s eta 0:00:00


In [30]:
#creating embeddings to be stored in the vector store
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)

In [31]:
query="What investments did AxiOraa make in Artifical Intelligence?"

In [34]:
#performing similarity search for performing effective retrieval
results=vector_store.similarity_search_with_score(
    query,
    k=3 #number of possible outcomes i.e. will give 3 most suitable answers to the query
)

for i, (doc, score) in enumerate(results, start=1):
  print(f"RESULT{i}")
  print(f"SIMILARITY SCORE:{score}")
  print(doc.page_content[:1000])

RESULT1
SIMILARITY SCORE:0.6495993137359619
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
RESULT2
SIMILARITY SCORE:0.6

In [35]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

print(retriever)

tags=['FAISS', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78200993b890> search_kwargs={'k': 3}


In [36]:
new_query="Summarize the AI Investments made by Axioraa"

retrieved_docs=retriever.invoke(new_query)

print(f"Retrieved {len(retrieved_docs)} Documents \n")

for i, doc in enumerate(retrieved_docs,start=1):
  print(f'DOCUMENT{i}')
  print(doc.page_content[:1000])

Retrieved 3 Documents 

DOCUMENT1
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
DOCUMENT2
s

In [38]:
#creating our RAG Prompt
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior Management Consultant,
    answer the user's question only using the retrieved context.

    If the answer cannot be found in the context, say: "I couldn't find this information"

    Retrieved Context {context}

    Question {question}

    Provide:

    1. Answer
    2. Supporting Evidence

    """
)

In [39]:
rag_chain=(
    rag_prompt | llm | StrOutputParser()
)

In [40]:
#Executing our RAG Query
question ="Summarize the AI Investments made by Axioraa"

retrieved_docs=retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke({
    "context":context,
    "question":question
})

print(response)

1. Answer:  
AxiOraa Ltd. has increased its investments in responsible AI and focuses on delivering AI strategy, data engineering, cloud modernization, and GenAI solutions across various sectors.

2. Supporting Evidence:  
The retrieved context mentions "recurring AI platform revenue and increased investments in responsible AI" and highlights that AxiOraa Ltd. delivers "AI strategy, data engineering, cloud modernization and GenAI solutions" to clients in banking, insurance, healthcare, manufacturing, and the public sector globally.


In [41]:
question ="Summarize the CEO's message"

retrieved_docs=retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke({
    "context":context,
    "question":question
})

print(response)

1. Answer  
The CEO's message highlights strong revenue growth in FY2025 driven by enterprise AI adoption. The company made significant investments in proprietary AI accelerators and strategic partnerships, which enhanced delivery efficiency despite macroeconomic uncertainty.

2. Supporting Evidence  
- "FY2025 was marked by strong revenue growth driven by enterprise AI adoption."  
- "Investments in proprietary AI accelerators and strategic partnerships improved delivery efficiency despite macroeconomic uncertainty."


Test the RAG Model with the following questions

1. How did the revenue perform during FY2025?
2. What risks were identified in the annual report?